# Hyperparameter Tuning for MRI-to-PET Diffusion Model
## Optuna Bayesian Optimization

This notebook runs Optuna hyperparameter search for the conditional diffusion model.

**Workflow:**
1. Run this notebook to find the best hyperparameters
2. Results are saved to `hp_study.json`
3. Copy the best parameters into the `Config` class in `MRI_to_PET_Diffusion_Model.ipynb`
4. Run the main notebook for full training with tuned parameters

**Search space:**

| Parameter | Range | Type |
|---|---|---|
| Learning rate | [5e-5, 5e-4] | Log uniform |
| Base channels | {48, 64} | Categorical |
| Channel mult | {(1,2,4,4), (1,2,4,8)} | Categorical |
| Dropout | [0.0, 0.15] | Float |
| Recon L1 weight | [0.0, 0.5] | Float |
| SSIM weight | [0.0, 0.5] | Float |
| Pyramid weight | [0.0, 0.3] | Float |
| Perceptual weight | [0.0, 0.3] | Float |
| CFG drop prob | [0.05, 0.2] | Float |
| CFG scale (infer) | [1.0, 4.0] | Float |
| Batch size | {4, 6} | Categorical |

Each trial trains for `HP_EPOCHS` epochs on `HP_DATA_FRAC` of training data, then evaluates SSIM.

In [ ]:
# ============================================================
# SETUP: Install Dependencies + Import Libraries
# ============================================================
!pip install optuna -q

import os, sys, gc, math, copy, random, time, json
from pathlib import Path
from datetime import datetime

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset
from torch.amp import autocast, GradScaler

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from skimage.metrics import structural_similarity as compare_ssim
from skimage.metrics import peak_signal_noise_ratio as compare_psnr
from scipy.ndimage import rotate as scipy_rotate
from scipy.ndimage import map_coordinates, gaussian_filter
import torchvision.models as tv_models
from tqdm.notebook import tqdm

import optuna
from optuna.trial import Trial
optuna.logging.set_verbosity(optuna.logging.WARNING)

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if device.type == "cuda":
    gpu = torch.cuda.get_device_properties(0)
    print(f"GPU: {gpu.name} | VRAM: {gpu.total_mem / 1e9:.1f} GB")

In [ ]:
# ============================================================
# CONFIGURATION — Base parameters for HP tuning
# ============================================================

class Config:
    # ---- Paths (CHANGE THESE) ----
    DATA_ROOT = "/kaggle/input/nacc-preprocessed"
    SAVE_DIR  = "/kaggle/working/hp_tuning"

    # ---- Data ----
    IMG_H, IMG_W    = 160, 192
    MIN_BRAIN_PX    = 800
    VAL_SPLIT       = 0.15

    # ---- Model (base values — these get overridden per trial) ----
    BASE_CH         = 64
    CH_MULT         = (1, 2, 4, 8)
    NUM_RES_BLOCKS  = 2
    ATTN_LEVELS     = (2, 3)
    TIME_EMB_DIM    = 256
    DROPOUT         = 0.05

    # ---- Diffusion ----
    NUM_TIMESTEPS   = 500
    SCHEDULE        = "cosine"

    # ---- Training (base values) ----
    BATCH_SIZE      = 6
    EPOCHS          = 200
    LR              = 5e-5
    WEIGHT_DECAY    = 1e-4
    EMA_DECAY       = 0.995
    GRAD_CLIP       = 1.0
    WARMUP_EPOCHS   = 10

    # ---- Hybrid Loss Weights (base values) ----
    W_NOISE         = 1.0
    W_RECON         = 0.5
    W_SSIM          = 0.2
    W_PYRAMID       = 0.2
    W_PERCEPTUAL    = 0.1

    # ---- Classifier-Free Guidance ----
    CFG_DROP_PROB   = 0.1
    CFG_SCALE       = 2.0

    # ---- Self-Conditioning ----
    SELF_COND_PROB  = 0.5

    # ---- Patch Training ----
    PATCH_SIZE      = 128

    # ---- Augmentation ----
    AUG_FLIP_P      = 0.5
    AUG_ROT_DEG     = 5.0
    AUG_ROT_P       = 0.3
    AUG_INTENSITY   = 0.05
    AUG_INTENSITY_P = 0.3
    AUG_NOISE_STD   = 0.02
    AUG_NOISE_P     = 0.2
    AUG_ELASTIC_P       = 0.3
    AUG_ELASTIC_ALPHA   = 80.0
    AUG_ELASTIC_SIGMA   = 10.0
    AUG_GAMMA_P         = 0.3
    AUG_GAMMA_RANGE     = (0.8, 1.2)
    AUG_BIAS_FIELD_P    = 0.2

    # ---- Sampling / Eval ----
    DDIM_STEPS      = 50
    DDIM_ETA        = 0.0
    NUM_VAL_SAMPLES = 100

    # ---- HP Tuning Settings ----
    HP_N_TRIALS     = 15
    HP_EPOCHS       = 5
    HP_DATA_FRAC    = 0.3
    HP_VAL_SAMPLES  = 50

cfg = Config()
os.makedirs(cfg.SAVE_DIR, exist_ok=True)

print("=" * 60)
print("HP TUNING CONFIGURATION")
print("=" * 60)
for attr in sorted(vars(Config)):
    if not attr.startswith('_'):
        print(f"  {attr:20s} = {getattr(cfg, attr)}")
print("=" * 60)

In [ ]:
# ============================================================
# DATASET WITH AUGMENTATION + CLASSIFIER-FREE GUIDANCE
# ============================================================

def gamma_transform(image, gamma):
    """Apply gamma intensity transform. Image expected in [-1, 1]."""
    img_01 = np.clip((image + 1.0) / 2.0, 0, 1)
    img_01 = np.power(img_01, gamma)
    return (img_01 * 2.0 - 1.0).astype(np.float32)


def random_bias_field(shape, order=3):
    """Generate a smooth multiplicative bias field using low-order polynomial.
    Returns a field with values centered around 1.0."""
    H, W = shape
    y = np.linspace(-1, 1, H)
    x = np.linspace(-1, 1, W)
    yy, xx = np.meshgrid(y, x, indexing='ij')
    coeffs = np.random.normal(0, 0.03, size=(order + 1, order + 1))
    coeffs[0, 0] = 1.0
    field = np.zeros((H, W), dtype=np.float32)
    for i in range(order + 1):
        for j in range(order + 1):
            if i + j <= order:
                field += coeffs[i, j] * (yy ** i) * (xx ** j)
    return field


def augment_pair(mri_ctx, pet_sl, cfg):
    """Apply matched augmentations to MRI-PET pair."""
    if random.random() < cfg.AUG_FLIP_P:
        mri_ctx = np.flip(mri_ctx, axis=2).copy()
        pet_sl  = np.flip(pet_sl, axis=2).copy()

    if cfg.AUG_ROT_DEG > 0 and random.random() < cfg.AUG_ROT_P:
        angle = random.uniform(-cfg.AUG_ROT_DEG, cfg.AUG_ROT_DEG)
        for c in range(mri_ctx.shape[0]):
            mri_ctx[c] = scipy_rotate(mri_ctx[c], angle, reshape=False,
                                       order=1, mode='nearest')
        pet_sl[0] = scipy_rotate(pet_sl[0], angle, reshape=False,
                                  order=1, mode='nearest')

    if random.random() < cfg.AUG_ELASTIC_P:
        shape_2d = mri_ctx.shape[1:]
        dx = gaussian_filter(np.random.randn(*shape_2d), cfg.AUG_ELASTIC_SIGMA) * cfg.AUG_ELASTIC_ALPHA
        dy = gaussian_filter(np.random.randn(*shape_2d), cfg.AUG_ELASTIC_SIGMA) * cfg.AUG_ELASTIC_ALPHA
        y, x = np.meshgrid(np.arange(shape_2d[0]), np.arange(shape_2d[1]), indexing='ij')
        indices = [np.clip(y + dy, 0, shape_2d[0] - 1),
                   np.clip(x + dx, 0, shape_2d[1] - 1)]
        for c in range(mri_ctx.shape[0]):
            mri_ctx[c] = map_coordinates(mri_ctx[c], indices, order=1, mode='reflect').astype(np.float32)
        pet_sl[0] = map_coordinates(pet_sl[0], indices, order=1, mode='reflect').astype(np.float32)

    if cfg.AUG_INTENSITY > 0 and random.random() < cfg.AUG_INTENSITY_P:
        s = 1.0 + random.uniform(-cfg.AUG_INTENSITY, cfg.AUG_INTENSITY)
        mri_ctx = np.clip(mri_ctx * s, -1.0, 1.0).astype(np.float32)

    if cfg.AUG_NOISE_STD > 0 and random.random() < cfg.AUG_NOISE_P:
        noise = np.random.normal(0, cfg.AUG_NOISE_STD, mri_ctx.shape).astype(np.float32)
        mri_ctx = np.clip(mri_ctx + noise, -1.0, 1.0)

    if random.random() < cfg.AUG_GAMMA_P:
        gamma = random.uniform(cfg.AUG_GAMMA_RANGE[0], cfg.AUG_GAMMA_RANGE[1])
        for c in range(mri_ctx.shape[0]):
            mri_ctx[c] = gamma_transform(mri_ctx[c], gamma)

    if random.random() < cfg.AUG_BIAS_FIELD_P:
        bias = random_bias_field(mri_ctx.shape[1:])
        for c in range(mri_ctx.shape[0]):
            mri_ctx[c] = np.clip(mri_ctx[c] * bias, -1.0, 1.0).astype(np.float32)

    return mri_ctx, pet_sl


class MRIPETSliceDataset(Dataset):
    """Loads MRI-PET volume pairs and yields 2D axial slices with 2.5D context."""

    def __init__(self, patient_dirs, cfg, augment=True):
        self.cfg = cfg
        self.augment = augment
        self.patch_size = cfg.PATCH_SIZE if augment else None
        self.patients = []
        self.samples  = []

        for pdir in tqdm(patient_dirs, desc="Indexing"):
            mri_p = os.path.join(pdir, "mri_processed.npy")
            pet_p = os.path.join(pdir, "pet_processed.npy")
            if not (os.path.exists(mri_p) and os.path.exists(pet_p)):
                continue
            try:
                mri = np.load(mri_p, mmap_mode='r')
                if mri.shape != (cfg.IMG_H, cfg.IMG_W, 160):
                    continue
                pidx = len(self.patients)
                self.patients.append((mri_p, pet_p))
                D = mri.shape[2]
                for k in range(1, D - 1):
                    if np.sum(np.array(mri[:, :, k]) > -0.9) > cfg.MIN_BRAIN_PX:
                        self.samples.append((pidx, k))
            except Exception:
                continue

        print(f"  -> {len(self.patients)} patients, {len(self.samples)} valid slices")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        pidx, k = self.samples[idx]
        mri_p, pet_p = self.patients[pidx]
        mri = np.load(mri_p, mmap_mode='r')
        pet = np.load(pet_p, mmap_mode='r')

        mri_ctx = np.stack([
            np.array(mri[:, :, k - 1], dtype=np.float32),
            np.array(mri[:, :, k],     dtype=np.float32),
            np.array(mri[:, :, k + 1], dtype=np.float32),
        ], axis=0)
        pet_sl = np.array(pet[:, :, k], dtype=np.float32)[None]

        if self.augment:
            mri_ctx, pet_sl = augment_pair(mri_ctx.copy(), pet_sl.copy(), self.cfg)

        if self.augment and random.random() < self.cfg.CFG_DROP_PROB:
            mri_ctx = np.zeros_like(mri_ctx)

        if self.patch_size is not None and self.patch_size < mri_ctx.shape[1]:
            H, W = mri_ctx.shape[1], mri_ctx.shape[2]
            ps = self.patch_size
            top  = random.randint(0, H - ps)
            left = random.randint(0, W - ps)
            mri_ctx = mri_ctx[:, top:top+ps, left:left+ps].copy()
            pet_sl  = pet_sl[:, top:top+ps, left:left+ps].copy()

        return torch.from_numpy(mri_ctx.copy()), torch.from_numpy(pet_sl.copy())


# ---- Build datasets ----
print("Building datasets...")
root = Path(cfg.DATA_ROOT)
all_dirs = sorted([str(d) for d in root.iterdir() if d.is_dir()])
print(f"Found {len(all_dirs)} patient directories")

random.seed(SEED)
random.shuffle(all_dirs)
n_val = max(1, int(len(all_dirs) * cfg.VAL_SPLIT))
val_dirs, train_dirs = all_dirs[:n_val], all_dirs[n_val:]
print(f"Split: {len(train_dirs)} train / {len(val_dirs)} val patients")

train_dataset = MRIPETSliceDataset(train_dirs, cfg, augment=True)
val_dataset   = MRIPETSliceDataset(val_dirs,   cfg, augment=False)

val_loader = DataLoader(val_dataset, batch_size=cfg.BATCH_SIZE, shuffle=False,
                        num_workers=2, pin_memory=True, persistent_workers=True)

print(f"\nTrain: {len(train_dataset):,} slices")
print(f"Val:   {len(val_dataset):,} slices ({len(val_loader):,} batches)")

In [ ]:
# ============================================================
# MODEL BUILDING BLOCKS
# ============================================================

def _num_groups(channels, max_groups=32):
    for g in range(min(max_groups, channels), 0, -1):
        if channels % g == 0:
            return g
    return 1


class SinusoidalTimeEmbedding(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def forward(self, t):
        half = self.dim // 2
        emb = math.log(10000) / (half - 1)
        emb = torch.exp(torch.arange(half, device=t.device, dtype=torch.float32) * -emb)
        emb = t.float()[:, None] * emb[None, :]
        return torch.cat([torch.sin(emb), torch.cos(emb)], dim=-1)


class ResBlock(nn.Module):
    def __init__(self, in_ch, out_ch, time_dim, dropout=0.0):
        super().__init__()
        self.norm1  = nn.GroupNorm(_num_groups(in_ch), in_ch)
        self.conv1  = nn.Conv2d(in_ch, out_ch, 3, padding=1)
        self.t_proj = nn.Linear(time_dim, out_ch)
        self.norm2  = nn.GroupNorm(_num_groups(out_ch), out_ch)
        self.drop   = nn.Dropout(dropout)
        self.conv2  = nn.Conv2d(out_ch, out_ch, 3, padding=1)
        self.skip   = nn.Conv2d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()

    def forward(self, x, t_emb):
        h = F.silu(self.norm1(x))
        h = self.conv1(h)
        h = h + self.t_proj(F.silu(t_emb))[:, :, None, None]
        h = F.silu(self.norm2(h))
        h = self.drop(h)
        h = self.conv2(h)
        return h + self.skip(x)


class SelfAttention(nn.Module):
    def __init__(self, channels, num_heads=4):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim  = channels // num_heads
        self.norm = nn.GroupNorm(_num_groups(channels), channels)
        self.qkv  = nn.Conv2d(channels, channels * 3, 1)
        self.proj = nn.Conv2d(channels, channels, 1)
        nn.init.zeros_(self.proj.weight)
        nn.init.zeros_(self.proj.bias)

    def forward(self, x):
        B, C, H, W = x.shape
        h = self.norm(x)
        qkv = self.qkv(h).reshape(B, 3, self.num_heads, self.head_dim, H * W)
        qkv = qkv.permute(1, 0, 2, 4, 3)
        q, k, v = qkv[0], qkv[1], qkv[2]
        if hasattr(F, 'scaled_dot_product_attention'):
            out = F.scaled_dot_product_attention(q, k, v)
        else:
            scale = self.head_dim ** -0.5
            attn = (q @ k.transpose(-2, -1)) * scale
            attn = F.softmax(attn, dim=-1)
            out = attn @ v
        out = out.permute(0, 1, 3, 2).reshape(B, C, H, W)
        return x + self.proj(out)


class Downsample(nn.Module):
    def __init__(self, ch):
        super().__init__()
        self.conv = nn.Conv2d(ch, ch, 3, stride=2, padding=1)
    def forward(self, x):
        return self.conv(x)


class Upsample(nn.Module):
    def __init__(self, ch):
        super().__init__()
        self.conv = nn.Conv2d(ch, ch, 3, padding=1)
    def forward(self, x):
        return self.conv(F.interpolate(x, scale_factor=2, mode='nearest'))

print("Building blocks defined.")

In [ ]:
# ============================================================
# CONDITIONAL U-NET (with Self-Conditioning support)
# ============================================================

class ConditionalUNet(nn.Module):
    def __init__(self, in_channels=5, out_channels=1, base_ch=64,
                 ch_mult=(1, 2, 4, 8), num_res_blocks=2,
                 attn_levels=(2, 3), time_emb_dim=256, dropout=0.1):
        super().__init__()
        channels  = [base_ch * m for m in ch_mult]
        n_levels  = len(channels)
        t_dim     = time_emb_dim * 4

        self.time_embed = nn.Sequential(
            SinusoidalTimeEmbedding(time_emb_dim),
            nn.Linear(time_emb_dim, t_dim), nn.SiLU(),
            nn.Linear(t_dim, t_dim),
        )
        self.conv_in = nn.Conv2d(in_channels, channels[0], 3, padding=1)

        self.encoder    = nn.ModuleList()
        self.downsamples = nn.ModuleList()
        prev_ch = channels[0]
        for lv in range(n_levels):
            ch = channels[lv]
            blocks = nn.ModuleList()
            blocks.append(ResBlock(prev_ch, ch, t_dim, dropout))
            for _ in range(num_res_blocks - 1):
                blocks.append(ResBlock(ch, ch, t_dim, dropout))
            if lv in attn_levels:
                blocks.append(SelfAttention(ch, max(1, ch // 64)))
            self.encoder.append(blocks)
            prev_ch = ch
            if lv < n_levels - 1:
                self.downsamples.append(Downsample(ch))

        mid = channels[-1]
        self.mid_block1 = ResBlock(mid, mid, t_dim, dropout)
        self.mid_attn   = SelfAttention(mid, max(1, mid // 64))
        self.mid_block2 = ResBlock(mid, mid, t_dim, dropout)

        self.decoder   = nn.ModuleList()
        self.upsamples = nn.ModuleList()
        prev_ch = mid
        for lv in reversed(range(n_levels)):
            ch = channels[lv]
            blocks = nn.ModuleList()
            blocks.append(ResBlock(prev_ch + ch, ch, t_dim, dropout))
            for _ in range(num_res_blocks - 1):
                blocks.append(ResBlock(ch, ch, t_dim, dropout))
            if lv in attn_levels:
                blocks.append(SelfAttention(ch, max(1, ch // 64)))
            self.decoder.append(blocks)
            if lv > 0:
                self.upsamples.append(Upsample(ch))
            prev_ch = ch

        self.out_norm = nn.GroupNorm(_num_groups(channels[0]), channels[0])
        self.out_conv = nn.Conv2d(channels[0], out_channels, 3, padding=1)

    def forward(self, x, t, cond, x0_prev=None):
        if x0_prev is None:
            x0_prev = torch.zeros_like(x)
        h = self.conv_in(torch.cat([x, cond, x0_prev], dim=1))
        t_emb = self.time_embed(t)

        skips = []
        for lv, blocks in enumerate(self.encoder):
            for block in blocks:
                h = block(h, t_emb) if isinstance(block, ResBlock) else block(h)
            skips.append(h)
            if lv < len(self.downsamples):
                h = self.downsamples[lv](h)

        h = self.mid_block1(h, t_emb)
        h = self.mid_attn(h)
        h = self.mid_block2(h, t_emb)

        up_idx = 0
        for blocks in self.decoder:
            h = torch.cat([h, skips.pop()], dim=1)
            for block in blocks:
                h = block(h, t_emb) if isinstance(block, ResBlock) else block(h)
            if up_idx < len(self.upsamples):
                h = self.upsamples[up_idx](h)
                up_idx += 1

        return self.out_conv(F.silu(self.out_norm(h)))


def build_model(cfg):
    return ConditionalUNet(
        in_channels=5, out_channels=1,
        base_ch=cfg.BASE_CH, ch_mult=cfg.CH_MULT,
        num_res_blocks=cfg.NUM_RES_BLOCKS,
        attn_levels=tuple(cfg.ATTN_LEVELS),
        time_emb_dim=cfg.TIME_EMB_DIM, dropout=cfg.DROPOUT,
    )


def count_params(model):
    return sum(p.numel() for p in model.parameters())

_m = build_model(cfg)
print(f"Model parameters: {count_params(_m):,}")
del _m; gc.collect()

In [ ]:
# ============================================================
# DIFFUSION PROCESS + LOSS FUNCTIONS
# ============================================================

def cosine_beta_schedule(T, s=0.008):
    steps = T + 1
    x = torch.linspace(0, T, steps)
    ac = torch.cos(((x / T) + s) / (1 + s) * math.pi * 0.5) ** 2
    ac = ac / ac[0]
    betas = 1 - (ac[1:] / ac[:-1])
    return torch.clamp(betas, 0.0001, 0.9999)


class GaussianDiffusion:
    def __init__(self, T=500, schedule='cosine'):
        self.T = T
        if schedule == 'cosine':
            betas = cosine_beta_schedule(T)
        else:
            betas = torch.linspace(1e-4, 0.02, T)

        self.betas = betas
        alphas = 1.0 - betas
        self.alphas_cumprod = torch.cumprod(alphas, dim=0)
        self.sqrt_ac        = torch.sqrt(self.alphas_cumprod)
        self.sqrt_1m_ac     = torch.sqrt(1.0 - self.alphas_cumprod)
        self.ac_prev     = F.pad(self.alphas_cumprod[:-1], (1, 0), value=1.0)
        self.post_var    = betas * (1.0 - self.ac_prev) / (1.0 - self.alphas_cumprod)

    def q_sample(self, x0, t, noise=None):
        if noise is None:
            noise = torch.randn_like(x0)
        sa = self.sqrt_ac.to(x0.device)[t].view(-1, 1, 1, 1)
        sm = self.sqrt_1m_ac.to(x0.device)[t].view(-1, 1, 1, 1)
        return sa * x0 + sm * noise

    def predict_x0(self, x_t, t, eps_pred):
        sa = self.sqrt_ac.to(x_t.device)[t].view(-1, 1, 1, 1)
        sm = self.sqrt_1m_ac.to(x_t.device)[t].view(-1, 1, 1, 1)
        return ((x_t - sm * eps_pred) / sa.clamp(min=1e-8)).clamp(-1, 1)

    @torch.no_grad()
    def ddim_sample(self, model, cond, shape, steps=50, eta=0.0, cfg_scale=1.0):
        device = cond.device
        B = shape[0]
        times = torch.linspace(self.T - 1, 0, steps + 1).long()
        x = torch.randn(shape, device=device)
        x0_prev = torch.zeros(shape, device=device)

        for i in tqdm(range(len(times) - 1), desc="DDIM", leave=False):
            t_now  = times[i].item()
            t_next = times[i + 1].item()
            t_b = torch.full((B,), t_now, device=device, dtype=torch.long)

            if cfg_scale > 1.0:
                x_double = torch.cat([x, x], dim=0)
                t_double = torch.cat([t_b, t_b], dim=0)
                c_double = torch.cat([cond, torch.zeros_like(cond)], dim=0)
                x0_prev_double = torch.cat([x0_prev, x0_prev], dim=0)
                with autocast('cuda', enabled=(device.type == 'cuda')):
                    eps_double = model(x_double, t_double, c_double, x0_prev_double)
                eps_cond, eps_uncond = eps_double.chunk(2, dim=0)
                eps = eps_uncond + cfg_scale * (eps_cond - eps_uncond)
            else:
                with autocast('cuda', enabled=(device.type == 'cuda')):
                    eps = model(x, t_b, cond, x0_prev)

            a_t    = self.alphas_cumprod.to(device)[t_now]
            a_next = self.alphas_cumprod.to(device)[max(t_next, 0)]

            x0_pred = ((x - torch.sqrt(1 - a_t) * eps) / torch.sqrt(a_t)).clamp(-1, 1)
            x0_prev = x0_pred

            if t_next <= 0:
                x = x0_pred
            else:
                sigma = eta * torch.sqrt((1 - a_next) / (1 - a_t) * (1 - a_t / a_next))
                dir_xt = torch.sqrt(1 - a_next - sigma ** 2) * eps
                x = torch.sqrt(a_next) * x0_pred + dir_xt
                if eta > 0:
                    x = x + sigma * torch.randn_like(x)
        return x


def _gaussian_window(size, sigma, channels, device):
    coords = torch.arange(size, dtype=torch.float32, device=device) - size // 2
    g = torch.exp(-(coords ** 2) / (2 * sigma ** 2))
    g = g / g.sum()
    window = g.unsqueeze(1) @ g.unsqueeze(0)
    return window.unsqueeze(0).unsqueeze(0).repeat(channels, 1, 1, 1)

def ssim_loss(pred, target, window_size=11, data_range=2.0):
    C = pred.shape[1]
    window = _gaussian_window(window_size, 1.5, C, pred.device)
    pad = window_size // 2
    mu_p = F.conv2d(pred, window, padding=pad, groups=C)
    mu_t = F.conv2d(target, window, padding=pad, groups=C)
    mu_pp, mu_tt, mu_pt = mu_p ** 2, mu_t ** 2, mu_p * mu_t
    sig_pp = F.conv2d(pred * pred, window, padding=pad, groups=C) - mu_pp
    sig_tt = F.conv2d(target * target, window, padding=pad, groups=C) - mu_tt
    sig_pt = F.conv2d(pred * target, window, padding=pad, groups=C) - mu_pt
    C1 = (0.01 * data_range) ** 2
    C2 = (0.03 * data_range) ** 2
    ssim_map = ((2 * mu_pt + C1) * (2 * sig_pt + C2)) / \
               ((mu_pp + mu_tt + C1) * (sig_pp + sig_tt + C2))
    return 1.0 - ssim_map.mean()


def laplacian_pyramid_loss(pred, target, n_levels=4):
    total = 0.0
    weights = [2 ** (n_levels - 1 - i) for i in range(n_levels)]
    w_sum = sum(weights)
    for i in range(n_levels):
        total += F.l1_loss(pred, target) * weights[i]
        if i < n_levels - 1:
            pred   = F.avg_pool2d(pred, 2)
            target = F.avg_pool2d(target, 2)
    return total / w_sum


class VGGPerceptualLoss(nn.Module):
    def __init__(self):
        super().__init__()
        vgg = tv_models.vgg16(weights=tv_models.VGG16_Weights.IMAGENET1K_V1)
        self.features = nn.Sequential(*list(vgg.features[:16]))
        for p in self.features.parameters():
            p.requires_grad_(False)
        self.features.eval()
        self.register_buffer('mean', torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer('std',  torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))

    def forward(self, pred, target):
        pred_01   = (pred + 1.0) / 2.0
        target_01 = (target + 1.0) / 2.0
        pred_3ch   = pred_01.repeat(1, 3, 1, 1)
        target_3ch = target_01.repeat(1, 3, 1, 1)
        pred_3ch   = (pred_3ch - self.mean) / self.std
        target_3ch = (target_3ch - self.mean) / self.std
        with torch.no_grad():
            target_feat = self.features(target_3ch)
        pred_feat = self.features(pred_3ch)
        return F.l1_loss(pred_feat, target_feat)


vgg_loss_fn = VGGPerceptualLoss().to(device)


def compute_hybrid_loss(eps_pred, eps_true, x_t, x_0, t, diffusion, cfg):
    brain_mask = (x_0 > -0.9).float()
    mask_sum = brain_mask.sum().clamp(min=1.0)

    noise_diff_sq = (eps_pred - eps_true) ** 2
    noise_loss = (noise_diff_sq * brain_mask).sum() / mask_sum

    x0_pred = diffusion.predict_x0(x_t, t, eps_pred)
    ac_t = diffusion.alphas_cumprod.to(x_t.device)[t].view(-1, 1, 1, 1)

    l1_diff = F.l1_loss(x0_pred, x_0, reduction='none')
    recon_loss = (l1_diff * ac_t * brain_mask).sum() / mask_sum

    ssim_w = ac_t.mean().item()
    s_loss = ssim_loss(x0_pred, x_0) * ssim_w
    pyr_loss = laplacian_pyramid_loss(x0_pred, x_0) * ssim_w
    perc_loss = vgg_loss_fn(x0_pred, x_0) * ssim_w

    total = (cfg.W_NOISE       * noise_loss +
             cfg.W_RECON       * recon_loss +
             cfg.W_SSIM        * s_loss +
             cfg.W_PYRAMID     * pyr_loss +
             cfg.W_PERCEPTUAL  * perc_loss)

    return total, {
        'noise': noise_loss.item(),
        'recon': recon_loss.item(),
        'ssim':  s_loss.item(),
        'pyramid': pyr_loss.item(),
        'perceptual': perc_loss.item(),
        'total': total.item(),
    }


diffusion = GaussianDiffusion(cfg.NUM_TIMESTEPS, cfg.SCHEDULE)
print(f"Diffusion: {cfg.NUM_TIMESTEPS} timesteps, {cfg.SCHEDULE} schedule")

In [ ]:
# ============================================================
# TRAINING UTILITIES: EMA, train_one_epoch, evaluate
# ============================================================

class EMA:
    def __init__(self, model, decay=0.995):
        self.decay = decay
        self.model = copy.deepcopy(model)
        self.model.eval()
        for p in self.model.parameters():
            p.requires_grad_(False)

    @torch.no_grad()
    def update(self, model):
        for ema_p, p in zip(self.model.parameters(), model.parameters()):
            ema_p.data.mul_(self.decay).add_(p.data, alpha=1 - self.decay)


def train_one_epoch(model, loader, optimizer, diffusion, cfg, scaler=None):
    model.train()
    loss_accum = {k: 0.0 for k in ['total', 'noise', 'recon', 'ssim', 'pyramid', 'perceptual']}
    n_batches = 0

    for mri_cond, pet_target in tqdm(loader, desc="Train", leave=False):
        mri_cond   = mri_cond.to(device, non_blocking=True)
        pet_target = pet_target.to(device, non_blocking=True)
        B = pet_target.shape[0]
        t = torch.randint(0, diffusion.T, (B,), device=device)
        noise = torch.randn_like(pet_target)
        x_t = diffusion.q_sample(pet_target, t, noise)

        x0_prev = torch.zeros_like(pet_target)
        if random.random() < cfg.SELF_COND_PROB:
            with torch.no_grad():
                if scaler is not None:
                    with autocast('cuda'):
                        eps_prev = model(x_t, t, mri_cond, x0_prev)
                else:
                    eps_prev = model(x_t, t, mri_cond, x0_prev)
                x0_prev = diffusion.predict_x0(x_t, t, eps_prev).detach()

        optimizer.zero_grad(set_to_none=True)

        if scaler is not None:
            with autocast('cuda'):
                eps_pred = model(x_t, t, mri_cond, x0_prev)
                loss, breakdown = compute_hybrid_loss(
                    eps_pred, noise, x_t, pet_target, t, diffusion, cfg)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), cfg.GRAD_CLIP)
            scaler.step(optimizer)
            scaler.update()
        else:
            eps_pred = model(x_t, t, mri_cond, x0_prev)
            loss, breakdown = compute_hybrid_loss(
                eps_pred, noise, x_t, pet_target, t, diffusion, cfg)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), cfg.GRAD_CLIP)
            optimizer.step()

        for k, v in breakdown.items():
            loss_accum[k] += v
        n_batches += 1

    return {k: v / max(1, n_batches) for k, v in loss_accum.items()}


@torch.no_grad()
def evaluate_model(model, loader, diffusion, cfg, max_samples=100):
    model.eval()
    ssim_all, brain_ssim_all, psnr_all, mae_all = [], [], [], []
    count = 0

    for mri_cond, pet_target in tqdm(loader, desc="Eval", leave=False):
        if count >= max_samples:
            break
        mri_cond = mri_cond.to(device, non_blocking=True)
        syn_pet = diffusion.ddim_sample(
            model, mri_cond, pet_target.shape,
            steps=cfg.DDIM_STEPS, eta=cfg.DDIM_ETA, cfg_scale=cfg.CFG_SCALE)

        syn_np  = syn_pet.cpu().numpy()
        real_np = pet_target.numpy()

        for b in range(syn_np.shape[0]):
            if count >= max_samples:
                break
            s = syn_np[b, 0]
            r = real_np[b, 0]

            ssim_val = compare_ssim(r, s, data_range=2.0)
            ssim_all.append(ssim_val)

            _, ssim_map = compare_ssim(r, s, data_range=2.0, full=True)
            mask = r > -0.9
            if mask.sum() > 100:
                brain_ssim_all.append(ssim_map[mask].mean())

            psnr_all.append(compare_psnr(r, s, data_range=2.0))
            mae_all.append(np.mean(np.abs(r - s)))
            count += 1

    return {
        'ssim':       np.mean(ssim_all) if ssim_all else 0.0,
        'ssim_std':   np.std(ssim_all)  if ssim_all else 0.0,
        'brain_ssim': np.mean(brain_ssim_all) if brain_ssim_all else 0.0,
        'psnr':       np.mean(psnr_all) if psnr_all else 0.0,
        'mae':        np.mean(mae_all)  if mae_all  else 0.0,
        'n_samples':  count,
    }

print("Training utilities defined.")

In [ ]:
# ============================================================
# HYPERPARAMETER TUNING WITH OPTUNA
# ============================================================

def hp_objective(trial: Trial, train_dataset, val_loader, diffusion, base_cfg):
    """Optuna objective: train short, evaluate SSIM."""
    trial_cfg = copy.copy(base_cfg)
    trial_cfg.LR         = trial.suggest_float('lr', 5e-5, 5e-4, log=True)
    trial_cfg.BASE_CH    = trial.suggest_categorical('base_ch', [48, 64])
    ch_key               = trial.suggest_categorical('ch_mult', ['standard', 'wide'])
    trial_cfg.CH_MULT    = (1, 2, 4, 8) if ch_key == 'standard' else (1, 2, 4, 4)
    trial_cfg.DROPOUT    = trial.suggest_float('dropout', 0.0, 0.15)
    trial_cfg.W_RECON    = trial.suggest_float('w_recon', 0.0, 0.5)
    trial_cfg.W_SSIM     = trial.suggest_float('w_ssim', 0.0, 0.5)
    trial_cfg.W_PYRAMID  = trial.suggest_float('w_pyramid', 0.0, 0.3)
    trial_cfg.W_PERCEPTUAL = trial.suggest_float('w_perceptual', 0.0, 0.3)
    trial_cfg.CFG_DROP_PROB = trial.suggest_float('cfg_drop', 0.05, 0.2)
    trial_cfg.CFG_SCALE  = trial.suggest_float('cfg_scale', 1.0, 4.0)
    trial_cfg.BATCH_SIZE = trial.suggest_categorical('batch_size', [4, 6])

    n_sub = int(len(train_dataset) * base_cfg.HP_DATA_FRAC)
    indices = random.sample(range(len(train_dataset)), n_sub)
    sub_dataset = Subset(train_dataset, indices)
    sub_loader  = DataLoader(sub_dataset, batch_size=trial_cfg.BATCH_SIZE,
                             shuffle=True, num_workers=2, pin_memory=True,
                             drop_last=True, persistent_workers=True)

    model = build_model(trial_cfg).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=trial_cfg.LR,
                                  weight_decay=trial_cfg.WEIGHT_DECAY)
    ema = EMA(model, decay=trial_cfg.EMA_DECAY)
    scaler = GradScaler('cuda') if device.type == 'cuda' else None

    for ep in range(1, base_cfg.HP_EPOCHS + 1):
        losses = train_one_epoch(model, sub_loader, optimizer, diffusion,
                                 trial_cfg, scaler)
        ema.update(model)

        if ep == base_cfg.HP_EPOCHS:
            metrics = evaluate_model(ema.model, val_loader, diffusion,
                                     trial_cfg, base_cfg.HP_VAL_SAMPLES)
            trial.report(metrics['ssim'], ep)

        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

    del model, optimizer, ema, scaler
    gc.collect()
    torch.cuda.empty_cache()

    return metrics['ssim']


# ---- Run HP Tuning ----
print("=" * 70)
print(f"HYPERPARAMETER TUNING: {cfg.HP_N_TRIALS} trials, "
      f"{cfg.HP_EPOCHS} epochs each, {cfg.HP_DATA_FRAC*100:.0f}% data")
print("=" * 70)

study = optuna.create_study(
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=SEED),
    pruner=optuna.pruners.MedianPruner(n_startup_trials=3),
)
study.optimize(
    lambda trial: hp_objective(trial, train_dataset, val_loader, diffusion, cfg),
    n_trials=cfg.HP_N_TRIALS,
    show_progress_bar=True,
)

# ---- Show results ----
print("\n" + "=" * 70)
print("HP TUNING RESULTS")
print("=" * 70)
print(f"Best trial SSIM: {study.best_value:.4f}")
print("Best hyperparameters:")
for k, v in study.best_params.items():
    print(f"  {k:20s} = {v}")

# Top 5 trials
print("\nTop 5 trials:")
trials_sorted = sorted(study.trials, key=lambda t: t.value if t.value else 0,
                       reverse=True)
for i, t in enumerate(trials_sorted[:5]):
    if t.value is not None:
        print(f"  #{i+1} SSIM={t.value:.4f} | {t.params}")

# ---- Save study ----
hp_save_path = os.path.join(cfg.SAVE_DIR, 'hp_study.json')
with open(hp_save_path, 'w') as f:
    json.dump({
        'best_value': study.best_value,
        'best_params': study.best_params,
        'all_trials': [{
            'number': t.number, 'value': t.value, 'params': t.params
        } for t in study.trials if t.value is not None],
    }, f, indent=2)
print(f"\nStudy saved to: {hp_save_path}")

In [ ]:
# ============================================================
# RESULTS: Copy these values into MRI_to_PET_Diffusion_Model.ipynb Config
# ============================================================

bp = study.best_params

print("=" * 70)
print("COPY THESE VALUES INTO YOUR MAIN NOTEBOOK CONFIG:")
print("=" * 70)
print(f"    LR              = {bp['lr']}")
print(f"    BASE_CH         = {bp['base_ch']}")
ch_tuple = '(1, 2, 4, 8)' if bp['ch_mult'] == 'standard' else '(1, 2, 4, 4)'
print(f"    CH_MULT         = {ch_tuple}")
print(f"    DROPOUT         = {bp['dropout']}")
print(f"    W_RECON         = {bp['w_recon']}")
print(f"    W_SSIM          = {bp['w_ssim']}")
print(f"    W_PYRAMID       = {bp['w_pyramid']}")
print(f"    W_PERCEPTUAL    = {bp['w_perceptual']}")
print(f"    CFG_DROP_PROB   = {bp['cfg_drop']}")
print(f"    CFG_SCALE       = {bp['cfg_scale']}")
print(f"    BATCH_SIZE      = {bp['batch_size']}")
print("=" * 70)

# ---- Visualization ----
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Trial SSIM values
trial_values = [t.value for t in study.trials if t.value is not None]
trial_numbers = [t.number for t in study.trials if t.value is not None]
axes[0].bar(trial_numbers, trial_values, color='steelblue', alpha=0.8)
axes[0].axhline(y=study.best_value, color='r', ls='--', label=f'Best: {study.best_value:.4f}')
axes[0].set_xlabel('Trial Number')
axes[0].set_ylabel('SSIM')
axes[0].set_title('SSIM per Trial')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Cumulative best
cum_best = []
best_so_far = 0
for v in trial_values:
    best_so_far = max(best_so_far, v)
    cum_best.append(best_so_far)
axes[1].plot(trial_numbers, cum_best, 'o-', color='green')
axes[1].set_xlabel('Trial Number')
axes[1].set_ylabel('Best SSIM So Far')
axes[1].set_title('Cumulative Best SSIM')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(cfg.SAVE_DIR, 'hp_tuning_results.png'), dpi=150)
plt.show()

print(f"\nDone! Best SSIM: {study.best_value:.4f}")
print(f"Results saved to: {cfg.SAVE_DIR}")